In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import Imagenette, SVHN
import numpy as np

import logging

import importlib
import FL_sim
import models_to_train

importlib.reload(FL_sim)
importlib.reload(models_to_train)
from models_to_train import ResNetPLModel
from FL_sim import FLSimulator, save_grads_f_applied_on_grads


torch.set_float32_matmul_precision('high')

In [2]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]
#             )
#         ])
#     ) for s in ['train', 'val']]


dataset = [
    SVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

In [3]:
def test_model_training():
    from pytorch_lightning import Trainer
    model = ResNetPLModel(num_classes=10,
        resnet_version='resnet18', lr=0.005)
    training_dataloader = torch.utils.data.DataLoader(
        dataset[0], batch_size=14000, shuffle=True,
        num_workers=10, persistent_workers=True)
    test_dataloader = torch.utils.data.DataLoader(
        dataset[1], batch_size=14000,
        num_workers=2, persistent_workers=True)
    trainer = Trainer(max_epochs=10, accelerator='cuda', log_every_n_steps=len(training_dataloader)//2)
    trainer.fit(model, training_dataloader, test_dataloader)
# test_model_training()

In [4]:
# dataset = [torch.utils.data.Subset(d, list(range(200))) for d in dataset]

In [5]:
def pre_send_process(single_model_grad_agent):
    return single_model_grad_agent


def server_rec_process(all_encoded_model_grad):
    return all_encoded_model_grad

def applied_on_grads_before_optim(fl_model, args_for_f_on_grad_dict):
    pass

In [6]:
try:
    if hasattr(sim.shared_train_loader, '_shutdown_workers'):
        sim.shared_train_loader._shutdown_workers()
    if hasattr(sim.test_loader, '_shutdown_workers'):
        sim.test_loader._shutdown_workers()
except: pass

k = 2  # Number of workers

batch_size = 13000
# batch_size /= 50 # imagenet
# batch_size /= 3 # resnet 50
batch_size = int(batch_size)

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005,
    logging_disabled=True, applied_on_grads_before_optim=applied_on_grads_before_optim)

sim = FLSimulator(
    num_agents=k, communication_rounds=2, client_epochs_per_round=30,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_flag=False, pl_model=model,
    pre_send_process=pre_send_process, server_rec_process=server_rec_process
)

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
sim.run_simulation()


round 1/2 --------------------
  - reporting global model metrics
         test loss: 

D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028


2.367, test auc: 0.472
         train loss: 2.344, train auc: 0.501
     > training agent 1/2


D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\pytorch_lightning\trainer\configuration_validator.py:74: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


         test loss: 0.434, test auc: 0.987
         train loss: 0.313, train auc: 0.994
     > training agent 2/2
         test loss: 0.476, test auc: 0.988
         train loss: 0.396, train auc: 0.992

round 2/2 --------------------
  - reporting global model metrics
         test loss: 47.986, test auc: 0.509
         train loss: 44.453, train auc: 0.508
     > training agent 1/2
         test loss: 2.170, test auc: 0.596
         train loss: 2.199, train auc: 0.585
     > training agent 2/2
         test loss: 2.218, test auc: 0.545
         train loss: Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\IPython\core\interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\hosse\AppData\Local\Temp\ipykernel_27832\790260094.py", line 26, in <module>
    sim.run_simulation()
  File "D:\User\App Files\Projects\VUB-ACS-25_Thesis\FL_sim.py", line 305, in run_simulation
  File "D:\User\App Files\Projects\VUB-ACS-25_Thesis\FL_sim.py", line 44, in report_metric
    for i, batch in enumerate(dataloader)]
                    ^^^^^^^^^^^^^^^^^^^^^
  File "D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\torch\utils\data\dataloader.py", line 434, in __iter__
    self._iterator = self._get_iterator()
                     ^^^^^^^^^^^^^^^^^^^^
  File "D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\torch\utils\data\dataloader.py", line 387, in _get_iterator
    return _MultiProcessingDataLoaderIter(self)

In [7]:
# try:
#     if hasattr(sim.shared_train_loader, '_shutdown_workers'):
#         sim.shared_train_loader._shutdown_workers()
#     if hasattr(sim.test_loader, '_shutdown_workers'):
#         sim.test_loader._shutdown_workers()
# except: pass
#
# for i in range(0, 6):
#     model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005, logging_disabled=True,
#                           applied_on_grads_before_optim=save_grads_f_applied_on_grads)
#
#     # save_folder_path = f"experiments/exp_data/gradients_resnet/gradients_resnet_t{i}/"
#     save_folder_path = f"exp_dump//"
#     model.args_for_f_on_grad['save_folder_path'] = save_folder_path
#
#     model.load_state_dict(torch.load('experiments/exp_data/resnet18_svhn.pth', map_location='cpu'))
#     k = 2
#     batch_size = 13000
#     batch_size /= 6 # more batches as samples
#     batch_size = int(batch_size)
#
#     sim = FLSimulator(
#         num_agents=k, communication_rounds=2, client_epochs_per_round=30,
#         batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
#         aggregation_method='fedavg', non_iid_flag=False, pl_model=model,
#         pre_send_process=pre_send_process, server_rec_process=server_rec_process,
#     )
#
#     logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
#     sim.run_simulation()


round 1/2 --------------------
  - reporting global model metrics
         test loss: 2.349, test auc: 0.497
         train loss: 2.339, train auc: 0.501
     > training agent 1/2


D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\pytorch_lightning\trainer\call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


         test loss: 400.391, test auc: 0.494
         train loss: 300.149, train auc: 0.496
     > training agent 2/2


RuntimeError: [enforce fail at inline_container.cc:595] . unexpected pos 25010688 vs 25010576